# PowerPointNet

**ID** — Presentasi `.pptx`: slide, bentuk, tabel, chart native, dan konversi HTML → slide.
**EN** — `.pptx` presentations: slides, shapes, tables, native charts, and HTML → slides.

> Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.

Panduan lengkap / full guide: [`docs/PowerPointNet.md`](../docs/PowerPointNet.md) ·
[Bahasa Indonesia](../docs/id/PowerPointNet.md)

In [ ]:
// Build first:  dotnet build OfficeNet.sln -c Release

#r "../src/OfficeNet.Core/bin/Release/net10.0/Gravicode.OfficeNet.Core.dll"
#r "../src/PdfNet/bin/Release/net10.0/Gravicode.OfficeNet.PdfNet.dll"
#r "../src/PowerPointNet/bin/Release/net10.0/Gravicode.OfficeNet.PowerPointNet.dll"
#r "../src/OfficeNet.Rendering/bin/Release/net10.0/Gravicode.OfficeNet.Rendering.dll"

// Published instead? Swap the lines above for:
//   #r "nuget: Gravicode.OfficeNet, *"

In [ ]:
using OfficeNet.Rendering;
using Microsoft.DotNet.Interactive.Formatting;

// Renders a document and shows the first page inline, so a cell's effect is visible rather than
// described. Base64 in an <img> because the notebook has nowhere to serve a file from.
void Show(string path, int width = 520)
{
    var png = DocumentRenderer.RenderThumbnail(path, width);
    var data = Convert.ToBase64String(png);

    display(HTML($"<img src='data:image/png;base64,{data}' style='border:1px solid #ddd' />"));
}

var work = Path.Combine(Path.GetTempPath(), "officenet-notebook");
Directory.CreateDirectory(work);
string At(string name) => Path.Combine(work, name);

Console.WriteLine($"Berkas ditulis ke / files written to: {work}");

## Slide dan tema / Slides and themes

Mengubah warna tema menata ulang setiap bentuk yang memakainya — bentuk berwarna eksplisit tidak
ikut berubah. /
Changing a theme colour restyles every shape that uses it — a shape given an explicit colour does
not follow.

In [ ]:
using PowerPointNet;
using PowerPointNet.Charts;
using PowerPointNet.Shapes;
using OfficeNet.Core;
using OfficeNet.Core.Charts;
using OfficeNet.Core.Drawing;

var deck = Presentation.Create();
deck.Master!.SetThemeColor("accent1", OfficeColor.FromRgb(0x1F, 0x38, 0x64));

deck.AddTitleSlide("OfficeNet", "Word, Excel, PowerPoint, dan PDF untuk .NET 10");

deck.AddBulletSlide("Komponen", [
    "WordNet — python-docx",
    "ExcelNet — openpyxl + pandas",
    "PowerPointNet — python-pptx + PptxGenJS",
    "PdfNet — PyPDF2",
]);

$"{deck.SlideCount} slide, {deck.SlideWidth.Inches:0.###} x {deck.SlideHeight.Inches:0.###} inci"

## Chart native / Native charts

Chart di `.pptx` adalah data, bukan gambar: angkanya ikut serta dan tema menata ulang tampilannya. /
A chart in a `.pptx` is data, not a picture: the numbers travel with it and the theme restyles it.

In [ ]:
var slide = deck.AddSlide(2);
slide.SetTitle("Pendapatan per Wilayah");

slide.AddChart(new ChartData
{
    Type = ChartType.Column,
    Categories = ["Jakarta", "Bandung", "Surabaya", "Medan"],
    Series =
    [
        new ChartSeries("2025", [1120, 860, 740, 410]),
        new ChartSeries("2026", [1480, 1150, 905, 520]),
    ],
    ValueAxisTitle = "Juta Rupiah",
    ValueFormat = "#,##0",
    ShowDataLabels = true,
    Legend = LegendPosition.Bottom,
});

deck.Save(At("deck.pptx"));
Show(At("deck.pptx"), 640);

Chart bisa dibaca kembali — `GetData()` mengembalikan nilai cache, yang persis yang ditampilkan
pembaca. /
Charts read back — `GetData()` returns the cached values, exactly what a viewer displays.

In [ ]:
var chart = slide.Charts.First();
var data = chart.GetData();

Console.WriteLine($"Tipe      : {data.Type}");
Console.WriteLine($"Kategori  : {string.Join(", ", data.Categories)}");

foreach (var series in data.Series)
    Console.WriteLine($"  {series.Name}: {string.Join(", ", series.Values)}");

// Ubah tipenya tanpa menyusun ulang datanya.
chart.SetData(data with { Type = ChartType.Bar });
chart.GetData().Type

## Bentuk dan efek / Shapes and effects

In [ ]:
var canvas = deck.AddSlide(3);   // Blank

canvas.AddShape(ShapeGeometry.RoundedRectangle,
        Units.Inches(1), Units.Inches(1.4), Units.Inches(3.4), Units.Inches(1.6))
    .WithGradientFill(
        OfficeColor.FromRgb(0x1F, 0x38, 0x64),
        OfficeColor.FromRgb(0x63, 0x8E, 0xC6),
        GradientDirection.DiagonalDown)
    .WithShadow(blur: Units.Pt(10));

canvas.AddShape(ShapeGeometry.Ellipse,
        Units.Inches(5.2), Units.Inches(1.4), Units.Inches(2.2), Units.Inches(2.2))
    .WithFill(OfficeColor.FromRgb(0xE8, 0x71, 0x22))
    .WithOutline(OfficeColor.White, Units.Pt(3));

deck.Save(At("deck.pptx"));
Show(At("deck.pptx"), 640);

## HTML → slide

Fitur PptxGenJS yang paling dicari. Isi yang panjang dipecah ke slide lanjutan alih-alih meluber. /
The PptxGenJS feature people come for. Long content splits onto continuation slides rather than
overflowing.

In [ ]:
using PowerPointNet.Html;

var html = @"
<h1>Tinjauan Kuartal</h1>
<p>Disusun oleh <strong>Gravicode Studios</strong>, dipimpin oleh <em>Kang Fadhil</em>.</p>
<h2>Sorotan</h2>
<ul>
  <li>Pendapatan naik <strong>32%</strong>
    <ul><li>Jakarta memimpin pertumbuhan</li></ul>
  </li>
  <li>Biaya operasional turun 4%</li>
</ul>
<h2>Ringkasan Angka</h2>
<table>
  <tr><th>Wilayah</th><th>2025</th><th>2026</th></tr>
  <tr><td>Jakarta</td><td>1.120</td><td>1.480</td></tr>
  <tr><td>Bandung</td><td>860</td><td>1.150</td></tr>
</table>";

using (var fromHtml = HtmlToSlides.CreatePresentation(html, new HtmlSlideOptions
{
    TitleSlide = "Dari HTML ke PowerPoint",
    SubtitleSlide = "Satu panggilan, tanpa penyuntingan manual",
}))
{
    Console.WriteLine($"{fromHtml.SlideCount} slide dihasilkan dari HTML");
    fromHtml.Save(At("html.pptx"));
}

Show(At("html.pptx"), 640);

## Ekspor PDF / PDF export

Chart digambar oleh pengekspor dari data cache-nya. /
Charts are drawn by the exporter from their cached data.

In [ ]:
deck.SaveAsPdf(At("deck.pdf"));
Show(At("deck.pdf"), 640);

In [ ]:
deck.Dispose();